In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("revenue_leakage_dirty_dataset.csv")

print("Original Shape:", df.shape)
df.head()

Original Shape: (6080, 17)


,Order_ID,Order_Date,Customer_ID,Product,Category,Region,City,Quantity,Unit_Price,Sales,Discount_Percent,Cost_Per_Unit,Delivery_Cost,Return_Status,Payment_Method,Payment_Status,Order_Status
0,ORD101240,2025-02-26,CUST1382,Blender,Home Appliances,South,Bengaluru,2,3079.48,6158.96,5.0,2234.47,239.75,No,Net Banking,Paid,Cancelled
1,ORD105079,2026-05-18,CUST1255,Coffee Maker,Home Appliances,East,Kolkata,4,5035.35,20141.4,0.0,3123.68,167.27,No,Net Banking,Paid,Delivered
2,ORD100562,2025-08-20,CUST1794,Jacket,Fashion,South,Bengaluru,1,5128.57,5128.57,10.0,3393.61,43.26,No,Debit Card,Paid,Delivered
3,ORD104670,2025-03-17,CUST1084,Mixer,Home Appliances,West,Pune,1,4357.93,4357.93,20.0,NaN,321.66,No,Cash,Pending,Cancelled
4,ORD104677,2025-05-20,CUST1276,Desk,Furniture,South,Hyderabad,3,11280.68,33842.04,0.0,9572.28,296.85,No,Debit Card,Failed,Delivered


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6080 entries, 0 to 6079
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          6080 non-null   str    
 1   Order_Date        6080 non-null   str    
 2   Customer_ID       6044 non-null   str    
 3   Product           6080 non-null   str    
 4   Category          6080 non-null   str    
 5   Region            6080 non-null   str    
 6   City              6008 non-null   str    
 7   Quantity          6080 non-null   int64  
 8   Unit_Price        6080 non-null   float64
 9   Sales             6060 non-null   str    
 10  Discount_Percent  6020 non-null   float64
 11  Cost_Per_Unit     6000 non-null   str    
 12  Delivery_Cost     5988 non-null   str    
 13  Return_Status     6044 non-null   str    
 14  Payment_Method    6032 non-null   str    
 15  Payment_Status    6080 non-null   str    
 16  Order_Status      6080 non-null   str    
dtypes: flo

In [4]:
df.isnull().sum()

Order_ID             0
Order_Date           0
Customer_ID         36
Product              0
Category             0
Region               0
City                72
Quantity             0
Unit_Price           0
Sales               20
Discount_Percent    60
Cost_Per_Unit       80
Delivery_Cost       92
Return_Status       36
Payment_Method      48
Payment_Status       0
Order_Status         0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(58)

In [6]:
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (6022, 17)


In [7]:
text_columns = [
    "Order_ID", "Customer_ID", "Product", "Category",
    "Region", "City", "Return_Status",
    "Payment_Method", "Payment_Status", "Order_Status"
]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

In [8]:
df["Product"] = df["Product"].str.title()

df["Category"] = df["Category"].str.title()

df["Region"] = df["Region"].str.title()

df["Return_Status"] = df["Return_Status"].str.title()

df["Payment_Method"] = df["Payment_Method"].str.title()

df["Payment_Status"] = df["Payment_Status"].str.title()

df["Order_Status"] = df["Order_Status"].str.title()

In [9]:
df["Category"] = df["Category"].replace({
    "Elec": "Electronics",
    "Furn": "Furniture",
    "Home Appliance": "Home Appliances"
})

In [10]:
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

In [11]:
numeric_columns = [
    "Sales",
    "Cost_Per_Unit",
    "Delivery_Cost"
]

for col in numeric_columns:
    df[col] = df[col].astype(str).str.replace(",", "", regex=False)
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [12]:
df.dtypes

Order_ID                       str
Order_Date          datetime64[us]
Customer_ID                    str
Product                        str
Category                       str
Region                         str
City                           str
Quantity                     int64
Unit_Price                 float64
Sales                      float64
Discount_Percent           float64
Cost_Per_Unit              float64
Delivery_Cost              float64
Return_Status                  str
Payment_Method                 str
Payment_Status                 str
Order_Status                   str
dtype: object

In [13]:
df.isnull().sum()

Order_ID              0
Order_Date            0
Customer_ID          36
Product               0
Category              0
Region                0
City                 72
Quantity              0
Unit_Price            0
Sales                40
Discount_Percent     60
Cost_Per_Unit       100
Delivery_Cost       112
Return_Status        36
Payment_Method       48
Payment_Status        0
Order_Status          0
dtype: int64

In [14]:
df["Sales"] = df["Sales"].fillna(
    df["Quantity"] * df["Unit_Price"]
)

In [15]:
df["Sales"].isnull().sum()

np.int64(0)

In [16]:
df["Cost_Per_Unit"] = df["Cost_Per_Unit"].fillna(
    df.groupby("Product")["Cost_Per_Unit"].transform("median")
)

In [17]:
df["Cost_Per_Unit"].isnull().sum()

np.int64(0)

In [18]:
df["Delivery_Cost"] = df["Delivery_Cost"].fillna(
    df.groupby("Region")["Delivery_Cost"].transform("median")
)

In [19]:
df["Delivery_Cost"].isnull().sum()

np.int64(0)

In [20]:
df.isnull().sum()

Order_ID             0
Order_Date           0
Customer_ID         36
Product              0
Category             0
Region               0
City                72
Quantity             0
Unit_Price           0
Sales                0
Discount_Percent    60
Cost_Per_Unit        0
Delivery_Cost        0
Return_Status       36
Payment_Method      48
Payment_Status       0
Order_Status         0
dtype: int64

In [21]:
df["Customer_ID"] = df["Customer_ID"].fillna("Unknown")

df["City"] = df["City"].fillna("Unknown")

df["Discount_Percent"] = df["Discount_Percent"].fillna(0)

df["Return_Status"] = df["Return_Status"].fillna("No")

df["Payment_Method"] = df["Payment_Method"].fillna("Unknown")

In [22]:
df.isnull().sum()

Order_ID            0
Order_Date          0
Customer_ID         0
Product             0
Category            0
Region              0
City                0
Quantity            0
Unit_Price          0
Sales               0
Discount_Percent    0
Cost_Per_Unit       0
Delivery_Cost       0
Return_Status       0
Payment_Method      0
Payment_Status      0
Order_Status        0
dtype: int64

In [23]:
df[df["Quantity"] <= 0]

,Order_ID,Order_Date,Customer_ID,Product,Category,Region,City,Quantity,Unit_Price,Sales,Discount_Percent,Cost_Per_Unit,Delivery_Cost,Return_Status,Payment_Method,Payment_Status,Order_Status
150,ORD103309,2025-05-12,CUST1478,Desk,Furniture,West,Pune,-1,10956.98,54784.90,5.0,9582.47,419.45,Yes,Cash,Paid,Returned
414,ORD103709,2025-11-11,CUST1369,T-Shirt,Fashion,North,Delhi,-1,811.26,1622.52,10.0,655.16,81.53,No,Cash,Pending,Cancelled
580,ORD102517,2025-01-22,CUST1638,Headphones,Electronics,East,Patna,-1,3199.09,15995.45,30.0,2142.62,426.82,No,Upi,Paid,Delivered
934,ORD104765,2025-06-24,CUST1279,Shoes,Fashion,North,Lucknow,-1,3331.88,6663.76,0.0,2716.83,221.71,Yes,Debit Card,Paid,Returned
1238,ORD102229,2025-10-31,CUST1670,Tablet,Electronics,North,Delhi,-1,22185.27,22185.27,15.0,18724.80,240.40,No,Debit Card,Failed,Cancelled
1677,ORD102485,2025-10-31,CUST1005,Shoes,Fashion,West,Ahmedabad,-1,2878.89,8636.67,15.0,1987.02,387.08,Yes,Cash,Paid,Delivered
2034,ORD104198,2026-03-13,CUST1040,Bookshelf,Furniture,East,Bhubaneswar,-1,6574.56,6574.56,10.0,4038.90,212.96,No,Upi,Paid,Cancelled
2292,ORD104431,2025-01-27,CUST1105,Desk,Furniture,Central,Bhopal,-1,11886.95,23773.90,5.0,7542.08,496.78,No,Cash,Pending,Delivered
2502,ORD103056,2025-08-30,CUST1465,Desk,Furniture,East,Patna,-1,10802.32,10802.32,0.0,7641.96,489.57,No,Credit Card,Failed,Delivered
2633,ORD104228,2025-09-10,CUST1154,Desk,Furniture,West,Pune,-1,11459.64,34378.92,25.0,6957.66,331.38,No,Cash,Failed,Cancelled


In [24]:
(df["Quantity"] <= 0).sum()

np.int64(12)

In [25]:
median_qty = df.loc[df["Quantity"] > 0, "Quantity"].median()

df.loc[df["Quantity"] <= 0, "Quantity"] = median_qty

In [26]:
(df["Quantity"] <= 0).sum()

np.int64(0)

In [27]:
df[df["Discount_Percent"] < 0]

,Order_ID,Order_Date,Customer_ID,Product,Category,Region,City,Quantity,Unit_Price,Sales,Discount_Percent,Cost_Per_Unit,Delivery_Cost,Return_Status,Payment_Method,Payment_Status,Order_Status


In [28]:
df[df["Discount_Percent"] > 100]

,Order_ID,Order_Date,Customer_ID,Product,Category,Region,City,Quantity,Unit_Price,Sales,Discount_Percent,Cost_Per_Unit,Delivery_Cost,Return_Status,Payment_Method,Payment_Status,Order_Status
177,ORD105238,2025-12-30,CUST1663,Tablet,Electronics,South,Chennai,2,24578.56,49157.12,110.0,16149.180,63.640,No,Net Banking,Paid,Cancelled
524,ORD101822,2025-09-22,CUST1132,Blender,Home Appliances,North,Jaipur,2,3369.95,6739.90,110.0,1951.510,66.750,No,Cash,Paid,Delivered
939,ORD102280,2025-04-12,CUST1727,Laptop,Electronics,East,Kolkata,4,48203.96,192815.84,110.0,36831.660,302.080,No,Credit Card,Paid,Cancelled
1572,ORD103548,2026-01-02,CUST1750,Bookshelf,Furniture,West,Ahmedabad,1,6689.53,6689.53,110.0,3683.460,494.410,No,Cash,Pending,Delivered
2520,ORD104141,2025-11-26,CUST1393,Shoes,Fashion,West,Mumbai,1,2847.48,2847.48,110.0,1567.470,295.020,No,Net Banking,Failed,Delivered
3083,ORD103667,2026-02-23,CUST1767,T-Shirt,Fashion,West,Pune,5,782.04,3910.20,110.0,592.720,116.920,Yes,Upi,Paid,Delivered
3232,ORD101222,2025-03-08,CUST1310,Mixer,Home Appliances,North,Lucknow,2,3614.30,7228.60,110.0,2337.650,463.730,Yes,Upi,Pending,Returned
3668,ORD101327,2026-05-27,Unknown,Shoes,Fashion,West,Mumbai,2,3415.20,6830.40,110.0,2195.730,49.140,No,Credit Card,Pending,Cancelled
4023,ORD102946,2025-04-02,CUST1329,Laptop,Electronics,North,Jaipur,3,49274.42,147823.26,110.0,38569.305,187.740,No,Net Banking,Failed,Delivered
4381,ORD104286,2026-11-05,CUST1441,Blender,Home Appliances,East,Bhubaneswar,4,2947.46,11789.84,110.0,2478.280,336.210,No,Net Banking,Paid,Delivered


In [29]:
(df["Discount_Percent"] < 0).sum()

np.int64(0)

In [30]:
(df["Discount_Percent"] > 100).sum()

np.int64(12)

In [31]:
median_discount = df.loc[
    (df["Discount_Percent"] >= 0) &
    (df["Discount_Percent"] <= 100),
    "Discount_Percent"
].median()

df.loc[
    (df["Discount_Percent"] < 0) |
    (df["Discount_Percent"] > 100),
    "Discount_Percent"
] = median_discount

In [32]:
df["Discount_Percent"].min()

np.float64(0.0)

In [33]:
df["Discount_Percent"].max()

np.float64(30.0)

In [34]:
df[df["Delivery_Cost"] < 0]

,Order_ID,Order_Date,Customer_ID,Product,Category,Region,City,Quantity,Unit_Price,Sales,Discount_Percent,Cost_Per_Unit,Delivery_Cost,Return_Status,Payment_Method,Payment_Status,Order_Status
266,ORD105798,2025-03-19,CUST1647,Bookshelf,Furniture,East,Patna,1,7248.52,7248.52,30.0,6171.79,-50.0,Yes,Cash,Paid,Returned
587,ORD100494,2025-02-20,CUST1610,Smartwatch,Electronics,South,Hyderabad,1,7402.61,7402.61,30.0,6019.36,-50.0,No,Upi,Pending,Cancelled
2076,ORD102782,2025-03-03,CUST1305,Shoes,Fashion,South,Hyderabad,3,3493.93,10481.79,25.0,2738.47,-50.0,Yes,Cash,Pending,Returned
2199,ORD105059,2025-03-22,CUST1449,Bookshelf,Furniture,West,Mumbai,1,6365.83,6365.83,25.0,4958.69,-50.0,No,Upi,Paid,Cancelled
2590,ORD103506,2025-06-28,CUST1782,Desk,Furniture,West,Mumbai,1,13666.22,13666.22,15.0,7826.00,-50.0,No,Net Banking,Paid,Delivered
2743,ORD103300,2025-08-17,CUST1691,Mixer,Home Appliances,North,Delhi,4,4796.13,19184.52,30.0,3773.95,-50.0,No,Net Banking,Pending,Cancelled
2756,ORD102760,2025-09-26,CUST1296,Laptop,Furniture,North,Delhi,2,9543.76,19087.52,30.0,5899.49,-50.0,No,Upi,Failed,Delivered
3354,ORD105102,2025-10-28,CUST1327,Microwave,Home Appliances,East,Patna,3,11197.94,33593.82,5.0,7614.31,-50.0,No,Net Banking,Paid,Delivered
3736,ORD104502,2026-07-04,CUST1006,Backpack,Accessories,North,Lucknow,1,1854.07,1854.07,30.0,1565.08,-50.0,No,Upi,Paid,Delivered
4910,ORD104420,2026-10-05,CUST1174,Smartphone,Electronics,East,Kolkata,3,24483.67,73451.01,20.0,19317.47,-50.0,No,Credit Card,Paid,Delivered


In [35]:
(df["Delivery_Cost"] < 0).sum()

np.int64(11)

In [36]:
df.loc[df["Delivery_Cost"] < 0, "Delivery_Cost"] = np.nan

df["Delivery_Cost"] = df["Delivery_Cost"].fillna(
    df.groupby("Region")["Delivery_Cost"].transform("median")
)

In [37]:
(df["Delivery_Cost"] < 0).sum()

np.int64(0)

In [38]:
df["Delivery_Cost"].isnull().sum()

np.int64(0)

In [39]:
df[df["Cost_Per_Unit"] <= 0]

,Order_ID,Order_Date,Customer_ID,Product,Category,Region,City,Quantity,Unit_Price,Sales,Discount_Percent,Cost_Per_Unit,Delivery_Cost,Return_Status,Payment_Method,Payment_Status,Order_Status


In [40]:
(df["Cost_Per_Unit"] <= 0).sum()

np.int64(0)

In [41]:
df["Cost_Per_Unit"].isnull().sum()

np.int64(0)

In [42]:
df[df["Sales"] <= 0]

,Order_ID,Order_Date,Customer_ID,Product,Category,Region,City,Quantity,Unit_Price,Sales,Discount_Percent,Cost_Per_Unit,Delivery_Cost,Return_Status,Payment_Method,Payment_Status,Order_Status


In [43]:
(df["Sales"] <= 0).sum()

np.int64(0)

In [44]:
df[df["Unit_Price"] <= 0]

,Order_ID,Order_Date,Customer_ID,Product,Category,Region,City,Quantity,Unit_Price,Sales,Discount_Percent,Cost_Per_Unit,Delivery_Cost,Return_Status,Payment_Method,Payment_Status,Order_Status


In [45]:
(df["Unit_Price"] <= 0).sum()

np.int64(0)

In [46]:
(df["Unit_Price"] <= 0).sum()

np.int64(0)

In [47]:
df.duplicated().sum()

np.int64(4)

In [48]:
df = df.drop_duplicates()

In [49]:
df.duplicated().sum()

np.int64(0)

In [50]:
df.dtypes

Order_ID                       str
Order_Date          datetime64[us]
Customer_ID                    str
Product                        str
Category                       str
Region                         str
City                           str
Quantity                     int64
Unit_Price                 float64
Sales                      float64
Discount_Percent           float64
Cost_Per_Unit              float64
Delivery_Cost              float64
Return_Status                  str
Payment_Method                 str
Payment_Status                 str
Order_Status                   str
dtype: object

In [51]:
(df["Quantity"] <= 0).sum()

np.int64(0)

In [52]:
((df["Discount_Percent"] < 0) | (df["Discount_Percent"] > 100)).sum()

np.int64(0)

In [53]:
(df["Delivery_Cost"] < 0).sum()

np.int64(0)

In [54]:
(df["Cost_Per_Unit"] <= 0).sum()

np.int64(0)

In [55]:
(df["Sales"] <= 0).sum()

np.int64(0)

In [56]:
df.to_csv("revenue_leakage_cleaned.csv", index=False)

In [57]:
print("Clean dataset saved successfully!")

Clean dataset saved successfully!


In [58]:
df["Discount_Amount"] = (
    df["Sales"] * df["Discount_Percent"] / 100
)

In [59]:
df[["Sales", "Discount_Percent", "Discount_Amount"]].head()

,Sales,Discount_Percent,Discount_Amount
0,6158.96,5.0,307.948
1,20141.40,0.0,0.000
2,5128.57,10.0,512.857
3,4357.93,20.0,871.586
4,33842.04,0.0,0.000


In [60]:
df["Net_Revenue"] = (
    df["Sales"] - df["Discount_Amount"]
)

In [61]:
df[["Sales", "Discount_Amount", "Net_Revenue"]].head()

,Sales,Discount_Amount,Net_Revenue
0,6158.96,307.948,5851.012
1,20141.40,0.000,20141.400
2,5128.57,512.857,4615.713
3,4357.93,871.586,3486.344
4,33842.04,0.000,33842.040


In [62]:
df["Product_Cost"] = (
    df["Quantity"] * df["Cost_Per_Unit"]
)

In [63]:
df["Total_Cost"] = (
    df["Product_Cost"] + df["Delivery_Cost"]
)

In [64]:
df["Profit"] = (
    df["Net_Revenue"] - df["Total_Cost"]
)

In [70]:
(df["Net_Revenue"] == 0).sum()

np.int64(0)

In [65]:
df["Profit_Margin"] = (
    df["Profit"] / df["Net_Revenue"] * 100
)

In [69]:
df[df["Profit"] < 0][[
    "Order_ID",
    "Product",
    "Sales",
    "Discount_Percent",
    "Net_Revenue",
    "Total_Cost",
    "Profit"
]].head(10)

,Order_ID,Product,Sales,Discount_Percent,Net_Revenue,Total_Cost,Profit
6,ORD105378,Backpack,1630.79,15.0,1386.1715,1602.70,-216.5285
17,ORD100764,T-Shirt,793.49,15.0,674.4665,1086.71,-412.2435
18,ORD104882,Headphones,13744.52,30.0,9621.1640,10907.10,-1285.9360
21,ORD103518,Smartwatch,12999.42,30.0,9099.5940,10546.47,-1446.8760
22,ORD103036,Laptop,2033.04,30.0,1423.1280,1746.14,-323.0120
23,ORD104743,Backpack,1722.75,15.0,1464.3375,1591.30,-126.9625
28,ORD104409,Coffee Maker,23962.72,30.0,16773.9040,20589.42,-3815.5160
29,ORD105437,Coffee Maker,5659.18,30.0,3961.4260,4435.77,-474.3440
37,ORD105281,T-Shirt,1724.82,10.0,1552.3380,1615.73,-63.3920
51,ORD104061,Blender,3066.45,25.0,2299.8375,2410.13,-110.2925


In [71]:
df["Discount_Leakage"] = df["Discount_Amount"]

In [72]:
df[[
    "Sales",
    "Discount_Percent",
    "Discount_Amount",
    "Discount_Leakage"
]].head(10)

,Sales,Discount_Percent,Discount_Amount,Discount_Leakage
0,6158.96,5.0,307.9480,307.9480
1,20141.40,0.0,0.0000,0.0000
2,5128.57,10.0,512.8570,512.8570
3,4357.93,20.0,871.5860,871.5860
4,33842.04,0.0,0.0000,0.0000
5,49755.61,5.0,2487.7805,2487.7805
6,1630.79,15.0,244.6185,244.6185
7,15174.28,5.0,758.7140,758.7140
8,3515.21,20.0,703.0420,703.0420
9,5737.55,25.0,1434.3875,1434.3875


In [73]:
df.sort_values(
    "Discount_Leakage",
    ascending=False
)[[
    "Order_ID",
    "Product",
    "Sales",
    "Discount_Percent",
    "Discount_Leakage",
    "Net_Revenue",
    "Profit"
]].head(10)

,Order_ID,Product,Sales,Discount_Percent,Discount_Leakage,Net_Revenue,Profit
583,ORD102192,Laptop,315379.80,30.0,94613.9400,220765.8600,-30654.3600
6035,ORD105851,Laptop,291428.05,30.0,87428.4150,203999.6350,773.1650
787,ORD100077,Laptop,283901.30,30.0,85170.3900,198730.9100,37260.5200
4072,ORD100817,Laptop,270059.95,30.0,81017.9850,189041.9650,-12173.2450
1382,ORD103206,Laptop,313955.80,25.0,78488.9500,235466.8500,23468.6400
4175,ORD101888,Laptop,250744.40,30.0,75223.3200,175521.0800,-37822.4200
5126,ORD101483,Laptop,247117.76,30.0,74135.3280,172982.4320,-16209.9880
3222,ORD105490,Laptop,246562.16,30.0,73968.6480,172593.5120,-2065.9180
3968,ORD104086,Laptop,241313.72,30.0,72394.1160,168919.6040,32102.7440
4035,ORD104529,Laptop,284136.75,25.0,71034.1875,213102.5625,-10653.2075


In [74]:
high_discount_loss = df[
    (df["Discount_Percent"] >= 20) &
    (df["Profit"] < 0)
]

In [75]:
len(high_discount_loss)

1295

In [76]:
high_discount_loss[[
    "Order_ID",
    "Product",
    "Category",
    "Region",
    "Sales",
    "Discount_Percent",
    "Discount_Amount",
    "Net_Revenue",
    "Total_Cost",
    "Profit"
]].head(10)

,Order_ID,Product,Category,Region,Sales,Discount_Percent,Discount_Amount,Net_Revenue,Total_Cost,Profit
18,ORD104882,Headphones,Electronics,East,13744.52,30.0,4123.3560,9621.1640,10907.10,-1285.9360
21,ORD103518,Smartwatch,Electronics,East,12999.42,30.0,3899.8260,9099.5940,10546.47,-1446.8760
22,ORD103036,Laptop,Fashion,North,2033.04,30.0,609.9120,1423.1280,1746.14,-323.0120
28,ORD104409,Coffee Maker,Home Appliances,South,23962.72,30.0,7188.8160,16773.9040,20589.42,-3815.5160
29,ORD105437,Coffee Maker,Home Appliances,Central,5659.18,30.0,1697.7540,3961.4260,4435.77,-474.3440
51,ORD104061,Blender,Home Appliances,West,3066.45,25.0,766.6125,2299.8375,2410.13,-110.2925
53,ORD105616,Mixer,Home Appliances,Central,14186.58,30.0,4255.9740,9930.6060,10720.97,-790.3640
63,ORD102580,Office Chair,Furniture,North,19218.48,25.0,4804.6200,14413.8600,16690.58,-2276.7200
70,ORD102113,Backpack,Accessories,Central,1973.18,30.0,591.9540,1381.2260,1591.72,-210.4940
80,ORD101311,Tablet,Electronics,West,23560.86,20.0,4712.1720,18848.6880,20359.53,-1510.8420


In [77]:
high_discount_loss["Profit"].sum()

np.float64(-2156299.2655)

In [78]:
abs(high_discount_loss["Profit"].sum())

np.float64(2156299.2655)

In [79]:
product_profit = df.groupby("Product").agg(
    Total_Sales=("Sales", "sum"),
    Total_Revenue=("Net_Revenue", "sum"),
    Total_Cost=("Total_Cost", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Orders=("Order_ID", "count")
).reset_index()

In [80]:
product_profit = product_profit.sort_values(
    "Total_Profit",
    ascending=True
)

In [81]:
product_profit.head(10)

,Product,Total_Sales,Total_Revenue,Total_Cost,Total_Profit,Total_Orders
14,T-Shirt,777279.27,6.689799e+05,644714.510,24265.3530,354
0,Backpack,1642205.28,1.395507e+06,1294718.025,100789.2545,374
1,Blender,2409610.18,2.036604e+06,1805755.640,230848.2230,361
5,Headphones,3208852.21,2.725598e+06,2395514.665,330083.5905,376
11,Shoes,2848975.44,2.472785e+06,2139062.130,333723.3160,373
6,Jacket,3766563.38,3.157547e+06,2783372.625,374174.0900,356
9,Mixer,3813349.86,3.253565e+06,2848868.315,404696.5425,394
3,Coffee Maker,5028757.47,4.239328e+06,3628114.960,611213.5320,374
2,Bookshelf,5665024.30,4.800254e+06,4131063.480,669190.6865,359
13,Smartwatch,6566384.48,5.529784e+06,4780722.985,749061.0770,404


In [82]:
loss_products = product_profit[
    product_profit["Total_Profit"] < 0
]

loss_products

,Product,Total_Sales,Total_Revenue,Total_Cost,Total_Profit,Total_Orders


In [83]:
region_profit = df.groupby("Region").agg(
    Total_Sales=("Sales", "sum"),
    Total_Revenue=("Net_Revenue", "sum"),
    Total_Cost=("Total_Cost", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Orders=("Order_ID", "count")
).reset_index()

In [85]:
region_profit = region_profit.sort_values(
    "Total_Profit",
    ascending=True
)
region_profit

,Region,Total_Sales,Total_Revenue,Total_Cost,Total_Profit,Total_Orders
3,South,30265642.76,2.560915e+07,2.220830e+07,3.400851e+06,1225
0,Central,30054589.51,2.564187e+07,2.192285e+07,3.719019e+06,1165
4,West,31563449.48,2.682512e+07,2.290429e+07,3.920831e+06,1239
2,North,31635804.71,2.704440e+07,2.296801e+07,4.076392e+06,1222
1,East,31348643.86,2.661925e+07,2.250754e+07,4.111715e+06,1167


In [86]:
loss_regions = region_profit[
    region_profit["Total_Profit"] < 0
]

loss_regions

,Region,Total_Sales,Total_Revenue,Total_Cost,Total_Profit,Total_Orders


In [87]:
returned_orders = df[
    df["Return_Status"] == "Yes"
]

len(returned_orders)

1110

In [88]:
return_rate = (
    len(returned_orders) / len(df)
) * 100

return_rate

18.44466600199402

In [89]:
returned_orders["Net_Revenue"].sum()

np.float64(24285238.018)

In [94]:
return_by_product = df.groupby("Product").agg(
    Total_Orders=("Order_ID", "count"),
    Returned_Orders=("Return_Status", lambda x: (x == "Yes").sum()),
    Total_Profit=("Profit", "sum")
).reset_index()

In [95]:
return_by_product["Return_Rate"] = (
    return_by_product["Returned_Orders"]
    / return_by_product["Total_Orders"]
) * 100

In [96]:
return_by_product.columns

Index(['Product', 'Total_Orders', 'Returned_Orders', 'Total_Profit',
       'Return_Rate'],
      dtype='str')

In [97]:
return_by_product.sort_values(
    "Return_Rate",
    ascending=False
).head(10)

,Product,Total_Orders,Returned_Orders,Total_Profit,Return_Rate
4,Desk,370,86,1.331529e+06,23.243243
13,Smartwatch,404,85,7.490611e+05,21.039604
10,Office Chair,419,81,1.097525e+06,19.331742
15,Tablet,352,68,2.398505e+06,19.318182
11,Shoes,373,71,3.337233e+05,19.034853
6,Jacket,356,67,3.741741e+05,18.820225
7,Laptop,389,73,5.758483e+06,18.766067
3,Coffee Maker,374,69,6.112135e+05,18.449198
9,Mixer,394,70,4.046965e+05,17.766497
0,Backpack,374,66,1.007893e+05,17.647059


In [98]:
return_by_product.head()

,Product,Total_Orders,Returned_Orders,Total_Profit,Return_Rate
0,Backpack,374,66,1.007893e+05,17.647059
1,Blender,361,62,2.308482e+05,17.174515
2,Bookshelf,359,62,6.691907e+05,17.270195
3,Coffee Maker,374,69,6.112135e+05,18.449198
4,Desk,370,86,1.331529e+06,23.243243


In [99]:
product_return_profit = df.groupby("Product").agg(
    Total_Orders=("Order_ID", "count"),
    Returned_Orders=("Return_Status", lambda x: (x == "Yes").sum()),
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).reset_index()

In [100]:
product_return_profit["Return_Rate"] = (
    product_return_profit["Returned_Orders"]
    / product_return_profit["Total_Orders"]
) * 100

In [101]:
product_return_profit.sort_values(
    "Return_Rate",
    ascending=False
).head(10)

,Product,Total_Orders,Returned_Orders,Total_Sales,Total_Profit,Return_Rate
4,Desk,370,86,10864548.45,1.331529e+06,23.243243
13,Smartwatch,404,85,6566384.48,7.490611e+05,21.039604
10,Office Chair,419,81,9032325.80,1.097525e+06,19.331742
15,Tablet,352,68,18371004.35,2.398505e+06,19.318182
11,Shoes,373,71,2848975.44,3.337233e+05,19.034853
6,Jacket,356,67,3766563.38,3.741741e+05,18.820225
7,Laptop,389,73,46243593.27,5.758483e+06,18.766067
3,Coffee Maker,374,69,5028757.47,6.112135e+05,18.449198
9,Mixer,394,70,3813349.86,4.046965e+05,17.766497
0,Backpack,374,66,1642205.28,1.007893e+05,17.647059


In [102]:
high_return_products = product_return_profit[
    product_return_profit["Return_Rate"] >= 20
]

In [103]:
high_return_products

,Product,Total_Orders,Returned_Orders,Total_Sales,Total_Profit,Return_Rate
4,Desk,370,86,10864548.45,1331529.327,23.243243
13,Smartwatch,404,85,6566384.48,749061.077,21.039604


In [104]:
customer_profit = df.groupby("Customer_ID").agg(
    Total_Orders=("Order_ID", "count"),
    Total_Sales=("Sales", "sum"),
    Total_Revenue=("Net_Revenue", "sum"),
    Total_Cost=("Total_Cost", "sum"),
    Total_Profit=("Profit", "sum")
).reset_index()

In [105]:
customer_profit["Profit_Margin"] = (
    customer_profit["Total_Profit"]
    / customer_profit["Total_Revenue"]
) * 100

In [106]:
customer_profit.sort_values(
    "Total_Profit",
    ascending=True
).head(10)

,Customer_ID,Total_Orders,Total_Sales,Total_Revenue,Total_Cost,Total_Profit,Profit_Margin
285,CUST1286,15,664396.07,517455.8395,545285.550,-27829.7105,-5.378181
566,CUST1567,8,266517.73,208463.4930,230868.950,-22405.4570,-10.747904
527,CUST1528,13,449278.28,339790.8955,362150.830,-22359.9345,-6.580498
82,CUST1083,8,251442.58,185543.1500,205191.075,-19647.9250,-10.589410
77,CUST1078,7,345242.98,248566.6870,262288.960,-13722.2730,-5.520560
79,CUST1080,6,320842.45,262481.1320,273937.350,-11456.2180,-4.364587
306,CUST1307,4,322584.53,271321.1060,281102.710,-9781.6040,-3.605176
462,CUST1463,4,81106.80,59430.3820,67905.510,-8475.1280,-14.260598
705,CUST1706,8,325138.27,250916.4915,259073.920,-8157.4285,-3.251053
197,CUST1198,6,148962.71,112637.4460,120252.330,-7614.8840,-6.760526


In [107]:
high_sales_low_profit = customer_profit[
    (customer_profit["Total_Sales"] > customer_profit["Total_Sales"].median()) &
    (customer_profit["Total_Profit"] < customer_profit["Total_Profit"].median())
]

In [108]:
high_sales_low_profit.sort_values(
    "Total_Sales",
    ascending=False
).head(10)

,Customer_ID,Total_Orders,Total_Sales,Total_Revenue,Total_Cost,Total_Profit,Profit_Margin
285,CUST1286,15,664396.07,517455.8395,545285.55,-27829.7105,-5.378181
684,CUST1685,14,633786.92,506447.4650,507103.16,-655.6950,-0.129469
375,CUST1376,9,532194.28,393186.7005,393652.30,-465.5995,-0.118417
527,CUST1528,13,449278.28,339790.8955,362150.83,-22359.9345,-6.580498
383,CUST1384,7,412002.89,315437.3895,299691.43,15745.9595,4.991786
555,CUST1556,11,396780.56,293486.6280,279964.43,13522.1980,4.607432
657,CUST1658,9,383996.20,298894.9745,286352.72,12542.2545,4.196208
50,CUST1051,8,378671.38,292537.7685,290134.12,2403.6485,0.821654
77,CUST1078,7,345242.98,248566.6870,262288.96,-13722.2730,-5.520560
485,CUST1486,7,339963.44,269169.1145,267206.15,1962.9645,0.729268


In [109]:
discount_by_product = df.groupby("Product").agg(
    Total_Orders=("Order_ID", "count"),
    Total_Sales=("Sales", "sum"),
    Total_Discount=("Discount_Amount", "sum"),
    Average_Discount=("Discount_Percent", "mean"),
    Total_Profit=("Profit", "sum")
).reset_index()

In [110]:
discount_by_product.sort_values(
    "Total_Discount",
    ascending=False
).head(10)

,Product,Total_Orders,Total_Sales,Total_Discount,Average_Discount,Total_Profit
7,Laptop,389,46243593.27,7.103934e+06,15.565553,5.758483e+06
12,Smartphone,410,25201904.72,3.501416e+06,14.329268,3.677386e+06
15,Tablet,352,18371004.35,2.671355e+06,14.971591,2.398505e+06
4,Desk,370,10864548.45,1.631070e+06,15.175676,1.331529e+06
8,Microwave,353,9427751.86,1.413825e+06,14.929178,1.137333e+06
10,Office Chair,419,9032325.80,1.359691e+06,15.584726,1.097525e+06
13,Smartwatch,404,6566384.48,1.036600e+06,15.618812,7.490611e+05
2,Bookshelf,359,5665024.30,8.647701e+05,15.320334,6.691907e+05
3,Coffee Maker,374,5028757.47,7.894290e+05,15.534759,6.112135e+05
6,Jacket,356,3766563.38,6.090167e+05,16.109551,3.741741e+05


In [111]:
discount_by_product.sort_values(
    "Average_Discount",
    ascending=False
).head(10)

,Product,Total_Orders,Total_Sales,Total_Discount,Average_Discount,Total_Profit
6,Jacket,356,3766563.38,6.090167e+05,16.109551,3.741741e+05
13,Smartwatch,404,6566384.48,1.036600e+06,15.618812,7.490611e+05
10,Office Chair,419,9032325.80,1.359691e+06,15.584726,1.097525e+06
7,Laptop,389,46243593.27,7.103934e+06,15.565553,5.758483e+06
3,Coffee Maker,374,5028757.47,7.894290e+05,15.534759,6.112135e+05
2,Bookshelf,359,5665024.30,8.647701e+05,15.320334,6.691907e+05
4,Desk,370,10864548.45,1.631070e+06,15.175676,1.331529e+06
0,Backpack,374,1642205.28,2.466980e+05,15.080214,1.007893e+05
15,Tablet,352,18371004.35,2.671355e+06,14.971591,2.398505e+06
8,Microwave,353,9427751.86,1.413825e+06,14.929178,1.137333e+06


In [112]:
discount_problem_products = discount_by_product[
    (discount_by_product["Average_Discount"] >= 20) &
    (discount_by_product["Total_Profit"] < 0)
]

In [113]:
discount_problem_products

,Product,Total_Orders,Total_Sales,Total_Discount,Average_Discount,Total_Profit


In [114]:
delivery_by_region = df.groupby("Region").agg(
    Total_Orders=("Order_ID", "count"),
    Total_Sales=("Sales", "sum"),
    Total_Delivery_Cost=("Delivery_Cost", "sum"),
    Average_Delivery_Cost=("Delivery_Cost", "mean"),
    Total_Profit=("Profit", "sum")
).reset_index()

In [115]:
delivery_by_region.sort_values(
    "Total_Delivery_Cost",
    ascending=False
)

,Region,Total_Orders,Total_Sales,Total_Delivery_Cost,Average_Delivery_Cost,Total_Profit
4,West,1239,31563449.48,339303.925,273.853047,3.920831e+06
3,South,1225,30265642.76,334305.380,272.902351,3.400851e+06
2,North,1222,31635804.71,330265.650,270.266489,4.076392e+06
1,East,1167,31348643.86,311328.490,266.776769,4.111715e+06
0,Central,1165,30054589.51,308446.430,264.760884,3.719019e+06


In [116]:
delivery_by_region.sort_values(
    "Average_Delivery_Cost",
    ascending=False
)

,Region,Total_Orders,Total_Sales,Total_Delivery_Cost,Average_Delivery_Cost,Total_Profit
4,West,1239,31563449.48,339303.925,273.853047,3.920831e+06
3,South,1225,30265642.76,334305.380,272.902351,3.400851e+06
2,North,1222,31635804.71,330265.650,270.266489,4.076392e+06
1,East,1167,31348643.86,311328.490,266.776769,4.111715e+06
0,Central,1165,30054589.51,308446.430,264.760884,3.719019e+06


In [117]:
delivery_problem_regions = delivery_by_region[
    delivery_by_region["Total_Profit"] < 0
]

delivery_problem_regions

,Region,Total_Orders,Total_Sales,Total_Delivery_Cost,Average_Delivery_Cost,Total_Profit


In [118]:
df["Month"] = df["Order_Date"].dt.to_period("M").astype(str)

In [119]:
monthly_analysis = df.groupby("Month").agg(
    Total_Orders=("Order_ID", "count"),
    Total_Sales=("Sales", "sum"),
    Total_Discount=("Discount_Amount", "sum"),
    Total_Revenue=("Net_Revenue", "sum"),
    Total_Cost=("Total_Cost", "sum"),
    Total_Profit=("Profit", "sum")
).reset_index()

In [120]:
monthly_analysis["Profit_Margin"] = (
    monthly_analysis["Total_Profit"]
    / monthly_analysis["Total_Revenue"]
) * 100

In [121]:
monthly_analysis

,Month,Total_Orders,Total_Sales,Total_Discount,Total_Revenue,Total_Cost,Total_Profit,Profit_Margin
0,2025-01,302,7574112.49,1.086637e+06,6.487475e+06,5396545.245,1.090930e+06,16.815938
1,2025-02,291,7450642.47,1.098607e+06,6.352036e+06,5364034.530,9.880010e+05,15.554085
2,2025-03,310,8190071.46,1.311555e+06,6.878516e+06,6008949.115,8.695673e+05,12.641785
3,2025-04,338,8535272.77,1.332383e+06,7.202889e+06,6195658.660,1.007231e+06,13.983703
4,2025-05,347,8606280.93,1.106685e+06,7.499596e+06,6281498.330,1.218098e+06,16.242180
5,2025-06,308,7033118.25,1.006883e+06,6.026235e+06,5132745.215,8.934903e+05,14.826673
6,2025-07,333,9485445.96,1.319026e+06,8.166420e+06,6924530.235,1.241890e+06,15.207271
7,2025-08,306,8261534.46,1.186384e+06,7.075150e+06,5997854.650,1.077295e+06,15.226467
8,2025-09,293,7189712.15,1.068785e+06,6.120927e+06,5219888.525,9.010386e+05,14.720623
9,2025-10,341,9255078.60,1.428514e+06,7.826564e+06,6823727.725,1.002837e+06,12.813243


In [122]:
monthly_analysis.sort_values(
    "Total_Profit",
    ascending=True
).head(5)

,Month,Total_Orders,Total_Sales,Total_Discount,Total_Revenue,Total_Cost,Total_Profit,Profit_Margin
21,2026-10,69,1889539.86,302945.3110,1.586595e+06,1437743.190,148851.3590,9.381815
19,2026-08,71,1885894.98,327199.1095,1.558696e+06,1385699.370,172996.5005,11.098798
22,2026-11,76,2055264.71,374848.4870,1.680416e+06,1488834.840,191581.3830,11.400829
20,2026-09,61,1524326.65,168892.8100,1.355434e+06,1153096.745,202337.0950,14.927847
23,2026-12,67,1833378.75,277130.5960,1.556248e+06,1333679.170,222568.9840,14.301638


In [123]:
monthly_analysis.sort_values(
    "Total_Discount",
    ascending=False
).head(5)

,Month,Total_Orders,Total_Sales,Total_Discount,Total_Revenue,Total_Cost,Total_Profit,Profit_Margin
9,2025-10,341,9255078.60,1.428514e+06,7.826564e+06,6823727.725,1.002837e+06,12.813243
18,2026-07,280,7836312.30,1.335215e+06,6.501098e+06,5554054.570,9.470432e+05,14.567435
3,2025-04,338,8535272.77,1.332383e+06,7.202889e+06,6195658.660,1.007231e+06,13.983703
6,2025-07,333,9485445.96,1.319026e+06,8.166420e+06,6924530.235,1.241890e+06,15.207271
2,2025-03,310,8190071.46,1.311555e+06,6.878516e+06,6008949.115,8.695673e+05,12.641785


In [124]:
summary = {
    "Total Orders": len(df),
    "Total Sales": df["Sales"].sum(),
    "Total Discount": df["Discount_Amount"].sum(),
    "Total Net Revenue": df["Net_Revenue"].sum(),
    "Total Cost": df["Total_Cost"].sum(),
    "Total Profit": df["Profit"].sum(),
    "Average Profit Margin": df["Profit_Margin"].mean(),
    "Loss Making Orders": (df["Profit"] < 0).sum(),
    "Returned Orders": (df["Return_Status"] == "Yes").sum()
}

summary

{'Total Orders': 6018,
 'Total Sales': np.float64(154868130.32),
 'Total Discount': np.float64(23128339.6635),
 'Total Net Revenue': np.float64(131739790.6565),
 'Total Cost': np.float64(112510982.97999999),
 'Total Profit': np.float64(19228807.6765),
 'Average Profit Margin': np.float64(9.851622754443873),
 'Loss Making Orders': np.int64(1593),
 'Returned Orders': np.int64(1110)}

In [125]:
summary_df = pd.DataFrame(
    summary.items(),
    columns=["Metric", "Value"]
)

summary_df

,Metric,Value
0,Total Orders,6.018000e+03
1,Total Sales,1.548681e+08
2,Total Discount,2.312834e+07
3,Total Net Revenue,1.317398e+08
4,Total Cost,1.125110e+08
5,Total Profit,1.922881e+07
6,Average Profit Margin,9.851623e+00
7,Loss Making Orders,1.593000e+03
8,Returned Orders,1.110000e+03


In [126]:
total_loss = abs(
    df.loc[df["Profit"] < 0, "Profit"].sum()
)

total_loss

np.float64(2263230.8535)

In [127]:
high_discount_loss_amount = abs(
    df.loc[
        (df["Discount_Percent"] >= 20) &
        (df["Profit"] < 0),
        "Profit"
    ].sum()
)

high_discount_loss_amount

np.float64(2156299.2655)

In [128]:
df.to_csv(
    "revenue_leakage_final.csv",
    index=False
)

In [129]:
print("Final dataset saved successfully!")

Final dataset saved successfully!


In [130]:
df.shape

(6018, 25)

In [131]:
df.columns

Index(['Order_ID', 'Order_Date', 'Customer_ID', 'Product', 'Category',
       'Region', 'City', 'Quantity', 'Unit_Price', 'Sales', 'Discount_Percent',
       'Cost_Per_Unit', 'Delivery_Cost', 'Return_Status', 'Payment_Method',
       'Payment_Status', 'Order_Status', 'Discount_Amount', 'Net_Revenue',
       'Product_Cost', 'Total_Cost', 'Profit', 'Profit_Margin',
       'Discount_Leakage', 'Month'],
      dtype='str')